In [ ]:
# --- Final summary ---
print("Pipeline Summary")
print("=" * 70)
print(f"{'Sample':<8} {'ECC':<8} {'Score':<8} {'COMET cells':<14} {'Genes':<8} {'obsm keys'}")
print("-" * 70)

for sid in SAMPLES:
    ecc_ok = registration_results.get(sid, {}).get('ecc_converged', False)
    ecc_score = registration_results.get(sid, {}).get('ecc_score', 0)
    
    cr = cellbin_results.get(sid, {})
    adata = cr.get('integrated_adata')
    
    if adata is not None:
        n_cells = adata.shape[0]
        n_genes = adata.shape[1]
        obsm_keys = len(adata.obsm)
    else:
        n_cells = n_genes = obsm_keys = 0
    
    print(f"{sid:<8} {'OK' if ecc_ok else 'FAIL':<8} {ecc_score:<8.3f} "
          f"{n_cells:<14,} {n_genes:<8,} {obsm_keys}")

print("=" * 70)
print(f"\nOutputs saved to: {OUTPUT_BASE}")

## 7. Summary

Final AnnData structure per sample:
```
adata.X                        # STOmics gene expression (COMET cells x genes)
adata.obs                      # Cell metadata (centroid, area, labels)
adata.obsm['spatial']          # Coordinates in STOmics DAPI space (reference)
adata.obsm['spatial_he']       # Coordinates in H&E space
adata.obsm['spatial_msi_glycan']     # Coordinates in MSI glycan space
adata.obsm['spatial_msi_peptide']    # Coordinates in MSI peptide space
adata.obsm['spatial_msi_metabolite'] # Coordinates in MSI metabolite space
adata.obsm['spatial_comet_original'] # Original COMET pixel coordinates
adata.uns['sample_id']         # Sample ID
adata.uns['chip_id']           # STOmics chip ID
adata.uns['disease']           # Disease subtype
adata.uns['registration_method']     # 'qupath_ecc'
adata.uns['ecc_score']         # ECC alignment score
```

In [ ]:
# Modalities to warp coordinates into (besides STOmics DAPI which is the reference)
MODALITIES = ['he', 'msi_glycan', 'msi_peptide', 'msi_metabolite']

for sid, cr in cellbin_results.items():
    if cr['integrated_adata'] is None:
        print(f"{sid}: no integrated AnnData -- skipping coordinate warping")
        continue
    
    print(f"\n{'='*60}")
    print(f"Warping coordinates for {sid}")
    print(f"{'='*60}")
    
    adata = cr['integrated_adata']
    
    # COMET cell centroids in original COMET pixel space (from label GeoJSON)
    comet_centroids_orig, _ = geojson_centroids(load_geojson(
        DefaultPaths.label_geojson_path(sid)
    ))
    
    # STOmics space coordinates (already in adata from aggregation)
    stomics_coords = adata.obsm['spatial']
    print(f"  spatial (STOmics DAPI): {stomics_coords.shape}")
    
    # Load sample transforms
    try:
        sample_transforms = load_qupath_transforms(str(TRANSFORMS_JSON), sid)
    except KeyError:
        print(f"  WARNING: {sid} not in transforms JSON")
        continue
    
    available = list(sample_transforms['transforms'].keys())
    print(f"  Available modalities: {available}")
    
    for modality in MODALITIES:
        if modality not in sample_transforms['transforms']:
            print(f"  {modality}: not available")
            continue
        
        # The loaded matrix is the FORWARD transform: modality -> STOmics
        # We need STOmics -> modality, so invert it
        forward_matrix = sample_transforms['transforms'][modality]['matrix']
        stomics_to_modality = invert_affine(forward_matrix)
        
        warped_coords = apply_affine_to_coordinates(stomics_coords, stomics_to_modality)
        
        obsm_key = f'spatial_{modality}'
        adata.obsm[obsm_key] = warped_coords
        print(f"  {obsm_key}: {warped_coords.shape} "
              f"(range x=[{warped_coords[:,0].min():.0f}, {warped_coords[:,0].max():.0f}], "
              f"y=[{warped_coords[:,1].min():.0f}, {warped_coords[:,1].max():.0f}])")
    
    # Store original COMET coordinates (match by cell count)
    n_adata_cells = adata.shape[0]
    if len(comet_centroids_orig) >= n_adata_cells:
        adata.obsm['spatial_comet_original'] = comet_centroids_orig[:n_adata_cells]
        print(f"  spatial_comet_original: {adata.obsm['spatial_comet_original'].shape}")
    
    # Re-save with warped coordinates
    chip = SAMPLES[sid]
    h5ad_path = cr['sample_dir'] / f"{sid}_{chip}_comet_stomics_integrated.h5ad"
    adata.write_h5ad(str(h5ad_path))
    print(f"  Saved updated AnnData: {h5ad_path.name}")
    print(f"  AnnData obsm keys: {list(adata.obsm.keys())}")

print(f"\n{'='*60}")
print("Coordinate warping complete")
print(f"{'='*60}")

## 6. Warped Coordinates for All Modalities

Apply the affine transforms from `qupath_transforms.json` to generate warped coordinates for each modality. These can be stored in AnnData `.obsm` for downstream spatial analysis.

For each sample, we compute COMET cell centroids in:
- **STOmics DAPI space** (reference frame, from QuPath+ECC composed transform)
- **H&E space** (from QuPath H&E affine)
- **MSI glycan space** (from QuPath MSI affine)
- **MSI peptide space** (from QuPath MSI affine)
- **MSI metabolite space** (from QuPath MSI affine)

In [ ]:
# --- Concordance visualization: centroid overlay per sample ---
for sid, cr in concordance_results.items():
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    comet_xy = cr['comet_centroids']
    stomics_xy = cr['stomics_centroids']
    
    # Subsample for performance
    n_max = 10000
    if len(comet_xy) > n_max:
        idx_c = np.random.choice(len(comet_xy), n_max, replace=False)
        comet_sub = comet_xy[idx_c]
    else:
        comet_sub = comet_xy
    if len(stomics_xy) > n_max:
        idx_s = np.random.choice(len(stomics_xy), n_max, replace=False)
        stomics_sub = stomics_xy[idx_s]
    else:
        stomics_sub = stomics_xy
    
    # Panel 1: COMET cells only
    axes[0].scatter(comet_sub[:, 0], comet_sub[:, 1], s=0.5, alpha=0.3, c='magenta')
    axes[0].set_title(f'{sid} COMET cells ({len(comet_xy):,})')
    axes[0].set_aspect('equal')
    axes[0].invert_yaxis()
    
    # Panel 2: STOmics cells only
    axes[1].scatter(stomics_sub[:, 0], stomics_sub[:, 1], s=0.5, alpha=0.3, c='green')
    axes[1].set_title(f'{sid} STOmics cells ({len(stomics_xy):,})')
    axes[1].set_aspect('equal')
    axes[1].invert_yaxis()
    
    # Panel 3: Overlay
    axes[2].scatter(stomics_sub[:, 0], stomics_sub[:, 1], s=0.5, alpha=0.2, c='green', label='STOmics')
    axes[2].scatter(comet_sub[:, 0], comet_sub[:, 1], s=0.5, alpha=0.2, c='magenta', label='COMET')
    m = cr['metrics']
    axes[2].set_title(f'{sid} Overlay (median dist={m["median_distance"]:.1f}px)')
    axes[2].set_aspect('equal')
    axes[2].invert_yaxis()
    axes[2].legend(markerscale=10)
    
    plt.tight_layout()
    plt.savefig(cellbin_results[sid]['sample_dir'] / f"{sid}_concordance_centroids.png",
                dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
concordance_results = {}

for sid, cr in cellbin_results.items():
    chip = SAMPLES[sid]
    
    print(f"\n{'='*60}")
    print(f"Concordance: {sid}")
    print(f"{'='*60}")
    
    # Get COMET centroids from warped label GeoJSON
    comet_centroids, comet_labels = geojson_centroids(cr['warped_label'])
    
    # Get STOmics centroids — try adjusted cellbin, then raw cellbin, then borders
    stomics_centroids = None
    stomics_source = None
    
    for cellbin_name, cellbin_path_fn in [
        ('adjusted cellbin', DefaultPaths.stomics_cellbin_path),
        ('raw cellbin', DefaultPaths.stomics_raw_cellbin_path),
    ]:
        cellbin_path = cellbin_path_fn(chip)
        if not cellbin_path.exists():
            continue
        try:
            stomics_adata = load_stomics_cellbin_gef(cellbin_path)
            if stomics_adata.shape[0] < 10:
                print(f"  {cellbin_name}: only {stomics_adata.shape[0]} cells, skipping...")
                del stomics_adata
                continue
            if 'spatial' in stomics_adata.obsm:
                stomics_centroids = stomics_adata.obsm['spatial']
            else:
                stomics_centroids = np.column_stack([
                    stomics_adata.obs['x'].values, stomics_adata.obs['y'].values
                ])
            stomics_source = cellbin_name
            print(f"  Loaded STOmics from {cellbin_name}: {len(stomics_centroids):,} cells")
            del stomics_adata
            break
        except Exception as e:
            print(f"  {cellbin_name} failed: {e}")
            continue
    
    # Try cellbin borders as last resort
    if stomics_centroids is None:
        try:
            from comet.alignment_utils import load_stomics_cellbin_borders
            raw_path = DefaultPaths.stomics_raw_cellbin_path(chip)
            adj_path = DefaultPaths.stomics_cellbin_path(chip)
            for bp in [adj_path, raw_path]:
                if bp.exists():
                    borders = load_stomics_cellbin_borders(str(bp))
                    if borders['n_cells'] > 10:
                        stomics_centroids = borders['centroids']
                        stomics_source = f"cellbin borders ({bp.name})"
                        print(f"  Loaded STOmics from borders: {len(stomics_centroids):,} cells")
                        break
        except Exception as e:
            print(f"  Borders fallback failed: {e}")
    
    if stomics_centroids is None:
        print(f"  No usable STOmics cellbin found -- skipping concordance")
        continue
    
    print(f"  COMET cells:   {len(comet_centroids):,}")
    print(f"  STOmics cells: {len(stomics_centroids):,} (from {stomics_source})")
    
    # Alignment metrics (distance-based)
    metrics = compute_alignment_metrics(comet_centroids, stomics_centroids)
    print(f"\n  Alignment Quality: {metrics['quality']}")
    print(f"  Median distance to nearest STOmics cell: {metrics['median_distance']:.1f} px")
    print(f"  Within 30px: {metrics['pct_within_30px']:.1f}%")
    print(f"  Within 50px: {metrics['pct_within_50px']:.1f}%")
    
    # Bidirectional cell comparison
    comparison = compare_segmentation_cells(
        comet_centroids, stomics_centroids, max_distance=30
    )
    
    print(f"\n  Cell Concordance (30px threshold):")
    summary = comparison['summary']
    for category, count in summary.items():
        print(f"    {category}: {count:,}")
    
    concordance_results[sid] = {
        'metrics': metrics,
        'comparison': comparison,
        'comet_centroids': comet_centroids,
        'stomics_centroids': stomics_centroids,
        'stomics_source': stomics_source,
    }
    
    # Save metrics
    metrics_path = cr['sample_dir'] / f"{sid}_concordance_metrics.json"
    with open(metrics_path, 'w') as f:
        json.dump({**metrics, 'stomics_source': stomics_source,
                   'concordance_summary': {k: int(v) for k, v in summary.items()}}, f, indent=2)

# --- Summary table ---
if concordance_results:
    print(f"\n{'='*60}")
    print(f"{'Sample':<8} {'Source':<20} {'Quality':<12} {'Median(px)':<12} {'<30px%':<10} {'1:1':<10}")
    print("-" * 72)
    for sid, cr in concordance_results.items():
        m = cr['metrics']
        s = cr['comparison']['summary']
        n_1to1 = s.get('1:1', 0)
        print(f"{sid:<8} {cr['stomics_source']:<20} {m['quality']:<12} "
              f"{m['median_distance']:<12.1f} {m['pct_within_30px']:<10.1f} {n_1to1:<10,}")
    print(f"{'='*60}")

## 5. COMET vs STOmics Cellbin Concordance

Compare COMET cell segmentations (warped) with STOmics cellbin segmentations:
- 1:1 matches (mutual nearest neighbors within threshold)
- Fragmented cells (one COMET cell → multiple STOmics cells)
- Merged cells (multiple COMET cells → one STOmics cell)
- COMET-only / STOmics-only cells

In [ ]:
cellbin_results = {}

for sid, chip in SAMPLES.items():
    if sid not in registration_results:
        print(f"Skipping {sid} - no registration")
        continue
    
    res = registration_results[sid]
    sample_dir = res['sample_dir']
    matrix = res['composed_matrix']
    
    print(f"\n{'='*60}")
    print(f"Building COMET cellbin for {sid}")
    print(f"{'='*60}")
    
    info = ALL_SAMPLES.get(sid, {})
    
    # --- Step 1: Load and warp Label GeoJSON (phenotyped cells) ---
    label_path = DefaultPaths.label_geojson_path(sid)
    roi_path = DefaultPaths.roi_geojson_path(sid)
    
    if not label_path.exists():
        print(f"  WARNING: Label GeoJSON not found at {label_path}")
        continue
    
    warped_label_path = sample_dir / f"{sid}_warped_label.geojson"
    warped_roi_path = sample_dir / f"{sid}_warped_roi.geojson"
    
    # Warp label GeoJSON
    if not warped_label_path.exists():
        print(f"  Loading label GeoJSON: {label_path.name}")
        label_geojson = load_geojson(label_path)
        n_cells = len(label_geojson['features'])
        print(f"  {n_cells:,} phenotyped cells loaded")
        
        print(f"  Warping with composed affine...")
        warped_label = apply_affine_to_geojson(
            label_geojson, matrix, output_path=warped_label_path
        )
    else:
        print(f"  Loading cached warped label GeoJSON")
        try:
            warped_label = load_geojson(warped_label_path)
        except json.JSONDecodeError:
            print(f"  WARNING: Cached GeoJSON is corrupt, re-warping...")
            warped_label_path.unlink()
            label_geojson = load_geojson(label_path)
            warped_label = apply_affine_to_geojson(
                label_geojson, matrix, output_path=warped_label_path
            )
    
    # Warp ROI GeoJSON
    warped_roi = None
    if roi_path.exists():
        if not warped_roi_path.exists():
            print(f"  Loading ROI GeoJSON: {roi_path.name}")
            roi_geojson = load_geojson(roi_path)
            print(f"  {len(roi_geojson['features']):,} ROI features")
            warped_roi = apply_affine_to_geojson(
                roi_geojson, matrix, output_path=warped_roi_path
            )
        else:
            try:
                warped_roi = load_geojson(warped_roi_path)
            except json.JSONDecodeError:
                print(f"  WARNING: Cached ROI GeoJSON corrupt, re-warping...")
                warped_roi_path.unlink()
                roi_geojson = load_geojson(roi_path)
                warped_roi = apply_affine_to_geojson(
                    roi_geojson, matrix, output_path=warped_roi_path
                )
    
    n_warped = len(warped_label['features'])
    print(f"  {n_warped:,} warped cells")
    
    # --- Step 2: Rasterize to mask ---
    # Only need DAPI dimensions for mask shape — read metadata only, not full image
    stomics_dapi_path = DefaultPaths.stomics_dapi_path(chip)
    try:
        with tifffile.TiffFile(str(stomics_dapi_path)) as tif:
            page = tif.pages[0]
            mask_shape = (page.shape[0], page.shape[1]) if page.shape[0] > page.shape[1] or len(page.shape) == 2 else (page.shape[0], page.shape[1])
            # Handle multi-channel: shape could be (C, H, W) or (H, W)
            if len(page.shape) == 3:
                mask_shape = (page.shape[1], page.shape[2])
            else:
                mask_shape = (page.shape[0], page.shape[1])
        print(f"  STOmics DAPI dimensions: {mask_shape}")
    except Exception as e:
        print(f"  ERROR reading DAPI dimensions: {e}")
        continue
    
    mask_path = sample_dir / f"{sid}_comet_cell_mask.tif"
    phenotype_map = None
    if not mask_path.exists():
        print(f"  Rasterizing {n_warped:,} cells to {mask_shape} mask...")
        result = rasterize_geojson_to_mask(warped_label, mask_shape, output_path=mask_path)
        if isinstance(result, tuple):
            mask, phenotype_map = result
        else:
            mask = result
        n_in_mask = len(np.unique(mask)) - 1  # exclude 0
        print(f"  {n_in_mask:,} cells in mask ({n_in_mask/n_warped*100:.1f}%)")
    else:
        print(f"  Loading cached mask")
        mask = tifffile.imread(str(mask_path))
    
    # --- Step 3: Assign transcripts via mask lookup (Path B) ---
    tissue_gef_path = DefaultPaths.stomics_tissue_gef_path(chip)
    integrated_adata = None
    extracellular_df = None
    
    if tissue_gef_path.exists():
        print(f"  Assigning transcripts via mask lookup...")
        try:
            result = aggregate_transcripts_by_mask(
                str(tissue_gef_path), mask, warped_label,
                phenotype_map=phenotype_map
            )
            if isinstance(result, tuple):
                integrated_adata, extracellular_df = result
            else:
                integrated_adata = result
            
            print(f"  Integrated: {integrated_adata.shape[0]:,} cells x "
                  f"{integrated_adata.shape[1]:,} genes")
            print(f"  Median transcripts/cell: "
                  f"{integrated_adata.obs['n_transcripts'].median():.0f}")
            
            integrated_adata.uns['sample_id'] = sid
            integrated_adata.uns['chip_id'] = chip
            integrated_adata.uns['disease'] = info.get('disease', 'Unknown')
            integrated_adata.uns['registration_method'] = 'qupath_ecc'
            integrated_adata.uns['ecc_score'] = res['ecc_score']
            integrated_adata.uns['transcript_assignment'] = 'mask_lookup'
            
            h5ad_path = sample_dir / f"{sid}_{chip}_comet_stomics_integrated.h5ad"
            integrated_adata.write_h5ad(str(h5ad_path))
            print(f"  Saved: {h5ad_path.name}")
        except Exception as e:
            print(f"  ERROR in transcript assignment: {e}")
            import traceback; traceback.print_exc()
    else:
        print(f"  Tissue GEF not found at {tissue_gef_path}")
        print(f"  Falling back to nearest-neighbor aggregation from cellbin...")
        
        cellbin_path = DefaultPaths.stomics_cellbin_path(chip)
        if cellbin_path.exists():
            try:
                stomics_adata = load_stomics_cellbin_gef(cellbin_path)
                integrated_adata = aggregate_expression_per_comet_cell(
                    warped_label, stomics_adata,
                    method=AGGREGATION_METHOD, max_distance=MAX_DISTANCE
                )
                integrated_adata.uns['transcript_assignment'] = 'nearest_neighbor_fallback'
                h5ad_path = sample_dir / f"{sid}_{chip}_comet_stomics_integrated.h5ad"
                integrated_adata.write_h5ad(str(h5ad_path))
                print(f"  Fallback complete: {integrated_adata.shape}")
            except Exception as e:
                print(f"  ERROR: {e}")
    
    cellbin_results[sid] = {
        'warped_label': warped_label,
        'warped_roi': warped_roi,
        'warped_label_path': warped_label_path,
        'integrated_adata': integrated_adata,
        'sample_dir': sample_dir,
        'phenotype_map': phenotype_map,
    }

print(f"\n{'='*60}")
print(f"Cellbin complete: {len(cellbin_results)}/{len(SAMPLES)} samples")
for sid, cr in cellbin_results.items():
    ia = cr['integrated_adata']
    if ia is not None:
        method = ia.uns.get('transcript_assignment', '?')
        print(f"  {sid}: {ia.shape[0]:,} cells x {ia.shape[1]:,} genes ({method})")
    else:
        print(f"  {sid}: no integrated data")
print(f"{'='*60}")

## 4. Warp COMET GeoJSON & Build COMET Cellbin

For each sample:
1. Load COMET **label** GeoJSON (phenotyped cells from `geojson/` directory, NOT `geojson_output/`)
2. Also load **ROI** GeoJSON (tissue regions: Tumor, TME, Vessels, etc.)
3. Apply composed affine (QuPath + ECC) to warp both into STOmics space
4. Rasterize warped cell polygons to uint32 mask
5. Assign STOmics transcripts to COMET cells via mask lookup (Path B — preserves cell labels + phenotypes)

**Note:** Uses `aggregate_transcripts_by_mask()` (direct h5py transcript assignment) rather than the older `aggregate_expression_per_comet_cell()` (nearest-neighbor). This is ~12,000x faster and preserves original COMET cell labels.

In [ ]:
def make_overlay(comet_dapi_path, stomics_dapi_path, matrix, downsample=4):
    """Create magenta/green blend overlay at reduced resolution."""
    comet = tifffile.imread(str(comet_dapi_path))
    stomics = tifffile.imread(str(stomics_dapi_path))
    if stomics.ndim == 3:
        stomics = stomics[0]
    
    # Normalize to uint8
    for name, arr in [("comet", comet), ("stomics", stomics)]:
        pass  # keep raw for now
    
    def to_uint8(img):
        if img.dtype == np.uint8:
            return img
        vals = img[img > 0]
        if len(vals) == 0:
            return np.zeros_like(img, dtype=np.uint8)
        p1, p99 = np.percentile(vals, [1, 99])
        return np.clip((img.astype(np.float32) - p1) / max(p99 - p1, 1) * 255, 0, 255).astype(np.uint8)
    
    comet_u8 = to_uint8(comet)
    stomics_u8 = to_uint8(stomics)
    
    # Downsample
    D = downsample
    comet_s = cv2.resize(comet_u8, None, fx=1/D, fy=1/D, interpolation=cv2.INTER_AREA)
    stomics_s = cv2.resize(stomics_u8, None, fx=1/D, fy=1/D, interpolation=cv2.INTER_AREA)
    
    # Scale matrix for downsampled images
    M = matrix.copy()
    M[0, 2] /= D
    M[1, 2] /= D
    
    # Warp COMET into STOmics space
    warped = cv2.warpAffine(comet_s, M.astype(np.float32),
                            (stomics_s.shape[1], stomics_s.shape[0]),
                            flags=cv2.INTER_LINEAR, borderValue=0)
    
    # Equalize for visibility
    w_eq = cv2.equalizeHist(warped)
    s_eq = cv2.equalizeHist(stomics_s)
    
    h, w = stomics_s.shape[:2]
    blend = np.zeros((h, w, 3), dtype=np.uint8)
    blend[..., 0] = w_eq   # magenta = COMET
    blend[..., 1] = s_eq   # green = STOmics
    blend[..., 2] = w_eq   # magenta
    
    return blend

# Generate overlays
n_samples = len(registration_results)
fig, axes = plt.subplots(1, n_samples, figsize=(6*n_samples, 6))
if n_samples == 1:
    axes = [axes]

for idx, (sid, res) in enumerate(registration_results.items()):
    blend = make_overlay(
        res['comet_dapi_path'], res['stomics_dapi_path'],
        res['composed_matrix'], downsample=ECC_DOWNSAMPLE
    )
    axes[idx].imshow(cv2.cvtColor(blend, cv2.COLOR_BGR2RGB))
    ecc_str = f"ECC={res['ecc_score']:.3f}" if res['ecc_converged'] else "ECC failed"
    axes[idx].set_title(f"{sid}\n{ecc_str}", fontsize=12)
    axes[idx].axis('off')
    
    # Save individual overlay
    overlay_path = res['sample_dir'] / f"{sid}_registration_overlay.png"
    cv2.imwrite(str(overlay_path), blend)

plt.suptitle("COMET (magenta) → STOmics DAPI (green) — QuPath + ECC", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_BASE / "all_samples_registration_overview.png", dpi=150, bbox_inches='tight')
plt.show()
print("Overlays saved")

## 3. Registration Overlays & QC

Generate magenta/green blend overlays for visual QC. Images are downsampled for display.

In [ ]:
registration_results = {}

for sid, chip in SAMPLES.items():
    print(f"\n{'='*60}")
    print(f"Registering {sid} ({chip})")
    print(f"{'='*60}")
    
    sample_dir = OUTPUT_BASE / f"{sid}_{chip}"
    sample_dir.mkdir(parents=True, exist_ok=True)
    
    # Extract COMET DAPI if needed
    comet_dapi_path = sample_dir / f"{sid}_dapi.tif"
    comet_bs_path = DefaultPaths.comet_bs_path(sid)
    if not comet_dapi_path.exists():
        if comet_bs_path.exists():
            print(f"  Extracting COMET DAPI (this may take a while for 40GB+ files)...")
            extract_comet_dapi(comet_bs_path, comet_dapi_path)
        else:
            print(f"  WARNING: COMET BS not found at {comet_bs_path}")
            continue
    else:
        print(f"  COMET DAPI already extracted")
    
    # STOmics DAPI
    stomics_dapi_path = DefaultPaths.stomics_dapi_path(chip)
    if not stomics_dapi_path.exists():
        print(f"  WARNING: STOmics DAPI not found at {stomics_dapi_path}")
        continue
    
    # Run QuPath + ECC
    try:
        composed_matrix, ecc_score, converged = register_qupath_ecc(
            sid, str(TRANSFORMS_JSON),
            source_img_path=str(comet_dapi_path),
            target_img_path=str(stomics_dapi_path),
            modality='comet',
            warp_mode=ECC_WARP_MODE,
            downsample=ECC_DOWNSAMPLE,
        )
        
        registration_results[sid] = {
            'composed_matrix': composed_matrix,
            'ecc_score': ecc_score,
            'ecc_converged': converged,
            'comet_dapi_path': comet_dapi_path,
            'stomics_dapi_path': stomics_dapi_path,
            'sample_dir': sample_dir,
        }
        
        # Save matrix
        matrix_path = sample_dir / f"{sid}_composed_matrix.json"
        with open(matrix_path, 'w') as f:
            json.dump({
                'matrix': composed_matrix.tolist(),
                'ecc_score': ecc_score,
                'ecc_converged': converged,
                'method': 'qupath_ecc',
            }, f, indent=2)
        
        status = "CONVERGED" if converged else "FALLBACK (QuPath only)"
        print(f"  ECC: score={ecc_score:.4f}, {status}")
        
    except Exception as e:
        print(f"  ERROR: {e}")
        import traceback; traceback.print_exc()

print(f"\n{'='*60}")
print(f"Registration complete: {len(registration_results)}/{len(SAMPLES)} samples")
print(f"{'='*60}")

## 2. QuPath + ECC Registration

For each sample:
1. Load QuPath affine (auto-inverted from `config/qupath_transforms.json`)
2. Extract COMET DAPI from BS OME-TIFF
3. Pre-warp COMET DAPI with QuPath affine
4. Run ECC refinement (Euclidean: rotation + translation correction)
5. Compose final transform = QuPath affine + ECC correction

In [ ]:
# --- Check data availability ---
# Use label GeoJSON (phenotyped cells from MLD export) + ROI GeoJSON (tissue regions)
# These are in geojson/ directory, NOT geojson_output/ (old TIF mask exports without phenotypes)
print(f"{'Sample':<8} {'Chip':<12} {'COMET BS':<10} {'Label GJ':<10} {'ROI GJ':<10} {'DAPI':<10} {'Cellbin':<10}")
print("-" * 72)
for sid, chip in SAMPLES.items():
    info = ALL_SAMPLES.get(sid, {})
    bs = DefaultPaths.comet_bs_path(sid).exists()
    label_gj = DefaultPaths.label_geojson_path(sid).exists()
    roi_gj = DefaultPaths.roi_geojson_path(sid).exists()
    dapi = DefaultPaths.stomics_dapi_path(chip).exists()
    cb = DefaultPaths.stomics_cellbin_path(chip).exists()
    print(f"{sid:<8} {chip:<12} {'OK' if bs else 'MISS':<10} {'OK' if label_gj else 'MISS':<10} "
          f"{'OK' if roi_gj else 'MISS':<10} {'OK' if dapi else 'MISS':<10} "
          f"{'OK' if cb else 'MISS':<10}")

print(f"\nLabel GeoJSON dir: {DefaultPaths.label_geojson_path('SO1').parent}")
print(f"ROI GeoJSON dir:   {DefaultPaths.roi_geojson_path('SO1').parent}")

In [ ]:
OUTPUT_BASE

In [ ]:
# --- Configuration ---
TRANSFORMS_JSON = Path("../config/qupath_transforms.json")
OUTPUT_BASE = Path(r"T:/Sammy Data/projects/out/comet_stomics_alignment")
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

# Samples to process (SO1-SO6, skip SO2 no chip, SO3 doesn't exist)
SAMPLES = {
    'SO1': 'D03453A6',
    'SO4': 'C03027C4',
    'SO5': 'C03027F5',
    'SO6': 'C03036D6',
    'SO14': 'C03137D4',
    'SO15': 'C03137E6'
}

# ECC parameters
ECC_WARP_MODE = 'euclidean'
ECC_DOWNSAMPLE = 4
AGGREGATION_METHOD = 'nearest'  # or 'polygon' (slower but more precise)
MAX_DISTANCE = 50  # pixels, for nearest-neighbor mapping

# Load transforms and show what's available
all_transforms = load_qupath_transforms(str(TRANSFORMS_JSON))
print(f"Loaded transforms for: {list(all_transforms.keys())}")
for sid in SAMPLES:
    if sid in all_transforms:
        modalities = list(all_transforms[sid]['transforms'].keys())
        print(f"  {sid}: {modalities}")
    else:
        print(f"  {sid}: NOT IN CONFIG - need to add QuPath matrices")

In [ ]:
import sys, json, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tifffile

# Project imports
sys.path.insert(0, str(Path.cwd().parent))
from comet.alignment_utils import (
    ALL_SAMPLES, DefaultPaths,
    load_geojson, geojson_centroids,
    extract_comet_dapi,
    load_stomics_cellbin_gef,
    aggregate_expression_per_comet_cell,
    aggregate_transcripts_by_mask,
    rasterize_geojson_to_mask,
    map_comet_to_stomics_cells,
    compare_segmentation_cells,
    compute_alignment_metrics,
    plot_alignment_validation,
    annotate_with_comet_phenotypes,
)
from comet.registration import (
    load_qupath_transforms,
    register_qupath_affine,
    register_qupath_ecc,
    apply_affine_to_coordinates,
    apply_affine_to_geojson,
    warp_image_large,
    plot_registration_overlay,
    compute_registration_metrics,
    invert_affine,
    compose_affine_transforms,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

print("Imports OK")

## 1. Setup & Configuration

# Script 07: QuPath + ECC Alignment Pipeline
## COMET → STOmics → MSI Multi-Modal Registration & Integration

**Workflow:**
1. **Setup** — Configure samples, load QuPath affine transforms
2. **ECC Refinement** — Refine QuPath affine with OpenCV ECC (COMET DAPI → STOmics DAPI)
3. **Registration Overlays** — Visual QC of alignment quality
4. **COMET Cellbin** — Warp COMET GeoJSON → STOmics space, rasterize to cell mask, aggregate gene expression
5. **STOmics vs COMET Cellbin Concordance** — Compare cell segmentations
6. **Warped Coordinates for AnnData** — Export coordinates for all modalities (COMET, MSI glycan/peptide/metabolite, H&E)

**Inputs:**
- `config/qupath_transforms.json` — QuPath manual affine matrices
- COMET BS OME-TIFF (for DAPI extraction)
- STOmics DAPI registered TIF
- STOmics cellbin GEF
- COMET GeoJSON cell segmentations

**Outputs per sample:**
- Registration overlay PNGs
- Warped GeoJSON (COMET cells in STOmics space)
- Integrated AnnData (COMET cells × STOmics genes)
- Cell concordance metrics
- Warped coordinates for each modality